# पाठ १३ - Cognee ज्ञान ग्राफहरूसँग एजेन्ट मेमोरी


## सेटअप

यस नोटबुकले किनाराको मेमोरी भएका बुद्धिमान **कोडिङ सहायक** कसरी निर्माण गर्ने भनेर देखाउँछ [**Cognee**](https://www.cognee.ai/) ज्ञान ग्राफहरूको प्रयोग गरेर र **माइक्रोसफ्ट एजेन्ट फ्रेमवर्क** (MAF) सँग।

Cognee असंरचित पाठलाई संरचित, सोध्न मिल्ने ज्ञान ग्राफमा परिणत गर्छ जुन भेक्टर एम्बेडिङहरूले समर्थित हुन्छ — जसले तपाईंको एजेन्टलाई समृद्ध, सम्बन्ध-जानकारी लामो समयसम्मको मेमोरी दिन्छ।

### तपाईंले के सिक्नुहुनेछ
1. **ज्ञान ग्राफहरू निर्माण गर्ने**: विकासकर्ता प्रोफाइलहरू र उत्तम अभ्यासहरूलाई संरचित, सोध्न मिल्ने ज्ञानमा रूपान्तरण गर्ने।
2. **Cognee लाई MAF सँग एकीकृत गर्ने**: `@tool` कार्यहरू प्रयोग गरी MAF एजेन्टलाई Cognee को ज्ञान ग्राफ सोध्न दिने।
3. **सत्र-जानकारीयुक्त संवादहरू**: एउटै सत्रमा धेरै प्रश्नहरूको सन्दर्भ कायम राख्ने।
4. **दीर्घकालीन मेमोरी**: महत्त्वपूर्ण ज्ञानलाई सत्रहरूमा कायम राख्ने र नयाँ संवादहरूमा पुनः प्राप्त गर्ने।

### आवश्यकताहरू
- Python 3.9+
- स्थानीय रूपमा Redis चलेको (`docker run -d -p 6379:6379 redis`) सत्र व्यवस्थापनको लागि
- LLM API कुञ्जी (जस्तै OpenAI) — `.env` मा `LLM_API_KEY` सेट गर्नुहोस्
- `.env` मा `CACHING=true` (Cognee सत्रहरूको लागि आवश्यक)
- Azure AI Foundry परियोजना र डिप्लोइ गरिएको च्याट मोडेल
- `.env` मा `AZURE_AI_PROJECT_ENDPOINT` र `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI मा प्रमाणीकरण गरिएको (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजेन्ट मेमोरीका प्रकारहरू

यो नोटबुकले मुख्य पाठ १३ नोटबुकका तीनै प्रकारका मेमोरीहरूलाई अन्वेषण गर्छ, तर लामो अवधिको मेमोरी ब्याकएण्डको रूपमा Cognee प्रयोग गर्छ:

| मेमोरी प्रकार | मेकानिज्म | आयु |
|---|---|---|
| **कार्यरत** | `agent.create_session()` (MAF) | एउटा संवाद |
| **छोटो अवधिको** | Cognee सत्र क्यास (Redis) | एउटा सत्र |
| **लामो अवधिको** | Cognee ज्ञान ग्राफ + भेक्टरहरू | स्थायी |

### Cognee को मेमोरी वास्तुकला
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## कग्नी स्टोरेज तयार गर्नुहोस्


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## भाग १ — ज्ञान आधार निर्माण

हामी हाम्रो कोडिङ सहायकका लागि व्यापक ज्ञान आधार बनाउन तीन प्रकारको डाटा लिन्छौं:

1. **डेवलपर प्रोफाइल** — व्यक्तिगत विशेषज्ञता र प्राविधिक पृष्ठभूमि
2. **पाइथन उत्कृष्ट अभ्यासहरू** — व्यवहारिक दिशानिर्देशहरूसहित पाइथनको ज़ेन
3. **ऐतिहासिक संवादहरू** — विकासकर्ता र एआई सहायकहरू बीचको अघिल्लो प्रश्नोत्तर सत्रहरू


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ज्ञान ग्राफ देखाउनुहोस्

Cognee ले निकालेका इकाइहरू र सम्बन्धहरूको अन्तरक्रियात्मक HTML दृश्यांकन प्रस्तुत गर्न सक्छ।


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## मेमोरीलाई Memify सँग समृद्ध बनाउनुहोस्

`memify()` ज्ञान ग्राफ विश्लेषण गर्दछ र बौद्धिक नियमहरू सिर्जना गर्दछ — पैटर्नहरू, उत्कृष्ट अभ्यासहरू, र अवधारणाहरू बीचको सम्बन्धहरू पहिचान गर्दै।


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## भाग २ — कग्नी उपकरणहरूसँग MAF एजेन्ट

अब हामी एउटा MAF एजेन्ट सिर्जना गर्दछौं जसले `@tool` कार्यहरू मार्फत कग्नीको ज्ञान ग्राफ सोध्न सक्छ। यसले एजेन्टलाई सत्रहरू मार्फत संवादात्मक सन्दर्भ कायम राख्दै ग्राफ-सचेत सेम्यान्टिक खोजको पूर्ण शक्ति प्रयोग गर्न दिन्छ।


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## सत्रहरूसँग कार्य स्मृति

`AgentSession` (`agent.create_session()` मार्फत सिर्जना गरिएको) सत्र भित्र कार्य स्मृति प्रदान गर्दछ। एजेन्टले पहिलेका सन्देशनहरूलाई फेरि हेर्न सक्छ भने त्यही समयमा Cognee को दीर्घकालीन ज्ञान ग्राफ पनि सोध्न सक्छ।


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## नयाँ सत्र — दीर्घकालीन स्मृति रहन्छ

नयाँ सत्र सुरु गर्दा कार्य स्मृति खाली हुन्छ, तर Cognee ज्ञान ग्राफ अझै उपलब्ध हुन्छ। एजेन्टले पूर्ण नयाँ कुराकानीमा पनि उस्तै दीर्घकालीन ज्ञान पुनःप्राप्त गर्न सक्छ।


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

यस नोटबुकमा तपाईले **MAF को कार्यक्षमता मेमोरी** (`agent.create_session()`) लाई **Cognee को दीर्घकालीन ज्ञान ग्राफ** सँग जोड्ने कोडिङ सहयोगी बनाउनु भयो।

### तपाईंले के सिक्नुभयो
1. **ज्ञान ग्राफ निर्माण**: Cognee ले असंरचित पाठ लिन्छ र ग्राफ + भेक्टर मेमोरी बनाउँछ।
2. **Memify संग ग्राफ समृद्धि**: तपाइँको विद्यमान ग्राफ माथि निकालेका तथ्यहरू र समृद्ध सम्बन्धहरू।
3. **MAF + Cognee एकीकरण**: `@tool` फंक्शनहरूले MAF एजेन्टलाई Cognee को ग्राफ सहज तरिकाले सोध्ने अनुमति दिन्छ।
4. **कार्यशील मेमोरी + दीर्घकालीन मेमोरी**: `AgentSession` (मार्फत `agent.create_session()`) ले सत्र सन्दर्भ प्रदान गर्दछ जबकि Cognee स्थायी ज्ञान दिन्छ।
5. **NodeSets सँग फिल्टर गरिएका खोजहरू**: ज्ञान ग्राफको विशिष्ट उपसमूहहरूलाई लक्षित गर्नुहोस् (जस्तै मात्र सिद्धान्तहरू)।

### मुख्य बुँदाहरू
- **Cognee** कच्चा पाठलाई संरचित, सम्बन्ध-जानकारी मेमोरीमा परिवर्तन गर्छ — एक फ्ल्याट भेक्टर स्टोर भन्दा बढी शक्तिशाली।
- **`@tool` फंक्शनहरू** MAF एजेन्टहरू र बाह्य ज्ञान प्रणालीहरूलाई सफा रूपमा जोड्छ।
- **`AgentSession`** (मार्फत `agent.create_session()`) प्रति-पहिचान सन्दर्भलाई दीर्घकालीन ज्ञानबाट अलग राख्छ।
- सोही ज्ञान ग्राफले बहु सत्रहरू र एजेन्टहरूलाई सेवा दिन्छ।

### वास्तविक संसारका अनुप्रयोगहरू
- **विकासकर्ता सहयोगीहरू**: कोड समीक्षा, घटना विश्लेषण, आर्किटेक्चर सहयोगीहरू
- **ग्राहक-सामना गर्ने सहयोगीहरू**: उत्पादन दस्तावेज, FAQs, र CRM नोटहरूमा आधारित समर्थन एजेन्टहरू
- **आन्तरिक विशेषज्ञ सहयोगीहरू**: नीति, कानुनी, वा सुरक्षा सहयोगीहरू जसले दिशानिर्देशहरूमा विचार गर्छन्
- **एकीकृत डाटा तहहरू**: संरचित र असंरचित डाटालाई एउटै खोजयोग्य ग्राफमा जोड्नुहोस्

### अर्को कदमहरू
- Cognee मा कालानुक्रमिक जागरूकता संग प्रयोग गर्नुहोस्
- डोमेन-विशिष्ट ग्राफ गुणस्तरका लागि OWL ओन्टोलोजी परिभाषित गर्नुहोस्
- समयसँगै पुनःप्राप्तिमा सुधार ल्याउन प्रयोगकर्ता प्रतिक्रिया लूप थप्नुहोस्
- एउटै Cognee मेमोरी तह साझेदारी गर्ने बहु-एजेन्ट प्रणालीहरूमा विस्तार गर्नुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यस दस्तावेजलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) मार्फत अनुवाद गरिएको हो। हामी शुद्धताको लागि प्रयासरत छौँ, तर कृपया जानकार हुनुहोस् कि स्वचालित अनुवादहरूमा त्रुटि वा असंगति हुन सक्छ। मूल दस्तावेज यसको मूल भाषामा अधिकृत स्राेत मानिनु पर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार हुनेछैनौँ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
